# Evaluasi Awal Model YOLO (YOLOv10, YOLO11, YOLO26)

Notebook ini dibuat untuk dijalankan di Google Colab menggunakan akselerasi GPU (Pilih Runtime -> Change runtime type -> T4 GPU). Notebook ini akan membandingkan berbagai model YOLO sesuai usulan rancangan.

In [ ]:
!pip install ultralytics pandas matplotlib opencv-python-headless

In [ ]:
import os
import time
import urllib.request
import pandas as pd
import matplotlib.pyplot as plt
from ultralytics import YOLO

# Download a sample video if it doesn't exist
VIDEO_URL = "https://github.com/intel-iot-devkit/sample-videos/raw/master/people-detection.mp4"
VIDEO_PATH = "sample_people.mp4"

if not os.path.exists(VIDEO_PATH):
    print(f"Downloading sample video to {VIDEO_PATH}...")
    urllib.request.urlretrieve(VIDEO_URL, VIDEO_PATH)
    print("Download complete.")

In [ ]:
MODELS_TO_TEST = [
    "yolov10n.pt",
    "yolov10s.pt",
    "yolo11n.pt",
    "yolo11s.pt",
    "yolo26n.pt",
    "yolo26s.pt"
]

results_data = []
print("Starting YOLO comparison on GPU...")

for model_name in MODELS_TO_TEST:
    print(f"\n--- Loading {model_name} ---")
    try:
        model = YOLO(model_name)
    except Exception as e:
        print(f"Failed to load {model_name}: {e}")
        continue
    
    print(f"Running inference for {model_name}...")
    start_time = time.time()
    # Run inference on device 0 (GPU)
    results = model.predict(source=VIDEO_PATH, save=False, stream=False, verbose=False, device=0)
    end_time = time.time()
    
    total_time = end_time - start_time
    total_frames = len(results)
    avg_fps = total_frames / total_time if total_time > 0 else 0
    
    total_people_detected = 0
    confidences = []
    
    for r in results:
        boxes = r.boxes
        if boxes is not None:
            person_boxes = boxes[boxes.cls == 0]
            total_people_detected += len(person_boxes)
            confidences.extend(person_boxes.conf.cpu().numpy().tolist())
    
    avg_conf = sum(confidences) / len(confidences) if confidences else 0
    
    print(f"Results for {model_name}:")
    print(f"  Frames Processed: {total_frames}")
    print(f"  Total Time: {total_time:.2f}s (Avg FPS: {avg_fps:.2f})")
    print(f"  Total Detections: {total_people_detected}")
    
    results_data.append({
        "Model": model_name,
        "Total_Frames": total_frames,
        "Total_Time_s": total_time,
        "Avg_FPS": avg_fps,
        "Total_Detections": total_people_detected,
        "Avg_Confidence": avg_conf
    })

df = pd.DataFrame(results_data)
df.to_csv("yolo_comparison_results.csv", index=False)
df

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('YOLO Models Comparison on T4 GPU', fontsize=16)

axes[0].bar(df['Model'], df['Avg_FPS'], color=['#4C72B0', '#DD8452', '#55A868', '#C44E52', '#8172B3', '#937860'][:len(df)])
axes[0].set_title('Average FPS (Higher is better)')
axes[0].set_ylabel('Frames Per Second')
for i, v in enumerate(df['Avg_FPS']):
    axes[0].text(i, v + 2, f"{v:.1f}", ha='center')

axes[1].bar(df['Model'], df['Avg_Confidence'], color=['#4C72B0', '#DD8452', '#55A868', '#C44E52', '#8172B3', '#937860'][:len(df)])
axes[1].set_title('Average Detection Confidence')
axes[1].set_ylabel('Confidence Score (0-1)')
axes[1].set_ylim(0, 1.0)
for i, v in enumerate(df['Avg_Confidence']):
    axes[1].text(i, v + 0.02, f"{v:.4f}", ha='center')

plt.tight_layout()
plt.savefig("yolo_comparison_plot.png", dpi=300)
plt.show()